# S3.3 — PDF Parser (Docling)

Interactive verification of the Docling-based PDF parser service.

- Section-aware parsing
- File validation (size, format, magic bytes)
- Async with timeout protection
- Batch parsing
- Factory + singleton pattern

In [1]:
import sys
sys.path.insert(0, "../..")

## 1. Pydantic Models

In [2]:
from src.schemas.pdf import Section, PDFContent

# Section model
s = Section(title="Abstract", content="We propose a new method.", level=1)
assert s.title == "Abstract"
assert s.level == 1
print(f"Section: {s.title} (level {s.level}), content length: {len(s.content)}")

# PDFContent model
c = PDFContent(
    raw_text="Full text here",
    sections=[s],
    tables=["col1|col2\nA|B"],
    figures=["Figure 1: Architecture"],
    page_count=12,
    parser_used="docling",
    parser_time_seconds=2.5,
)
assert len(c.sections) == 1
assert c.page_count == 12
print(f"PDFContent: {len(c.sections)} sections, {len(c.tables)} tables, {len(c.figures)} figures")
print("\n\u2713 Pydantic models validated")

Section: Abstract (level 1), content length: 24
PDFContent: 1 sections, 1 tables, 1 figures

✓ Pydantic models validated


## 2. File Validation

In [3]:
from pathlib import Path
import tempfile
from src.services.pdf_parser.service import PDFParserService
from src.exceptions import PDFValidationError

svc = PDFParserService(max_file_size_mb=50)

with tempfile.TemporaryDirectory() as td:
    td = Path(td)

    # Valid PDF
    valid = td / "valid.pdf"
    valid.write_bytes(b"%PDF-1.4 test content")
    svc._validate_file(valid)
    print("Valid PDF: passed")

    # Missing file
    try:
        svc._validate_file(td / "nope.pdf")
    except PDFValidationError as e:
        print(f"Missing file: caught {e.detail}")

    # Not a PDF
    txt = td / "readme.txt"
    txt.write_text("hello")
    try:
        svc._validate_file(txt)
    except PDFValidationError as e:
        print(f"Non-PDF ext: caught {e.detail}")

    # Bad magic bytes
    bad = td / "bad.pdf"
    bad.write_bytes(b"NOT A PDF")
    try:
        svc._validate_file(bad)
    except PDFValidationError as e:
        print(f"Bad magic: caught {e.detail}")

print("\n\u2713 File validation working correctly")

Valid PDF: passed
Missing file: caught File not found: /var/folders/09/369xpjkj0bx605z54hlxdxv80000gn/T/tmp77w3dlpi/nope.pdf
Non-PDF ext: caught Not a PDF file: /var/folders/09/369xpjkj0bx605z54hlxdxv80000gn/T/tmp77w3dlpi/readme.txt
Bad magic: caught Invalid PDF file: /var/folders/09/369xpjkj0bx605z54hlxdxv80000gn/T/tmp77w3dlpi/bad.pdf

✓ File validation working correctly


## 3. Factory & Settings

In [4]:
from src.services.pdf_parser.factory import make_pdf_parser_service, reset_pdf_parser_cache

reset_pdf_parser_cache()
svc = make_pdf_parser_service()

assert svc.max_pages == 30
assert svc.max_file_size_mb == 50
assert svc.timeout == 120
print(f"max_pages: {svc.max_pages}")
print(f"max_file_size_mb: {svc.max_file_size_mb}")
print(f"timeout: {svc.timeout}s")

# Singleton check
svc2 = make_pdf_parser_service()
assert svc is svc2
print("Singleton: same instance")

reset_pdf_parser_cache()
print("\n\u2713 Factory and settings verified")

max_pages: 30
max_file_size_mb: 50
timeout: 120s
Singleton: same instance

✓ Factory and settings verified


## 4. Config Integration

In [5]:
from src.config import get_settings

settings = get_settings()
assert hasattr(settings, "pdf_parser")
assert settings.pdf_parser.max_pages == 30
assert settings.pdf_parser.max_file_size_mb == 50
assert settings.pdf_parser.timeout == 120
print(f"PDFParserSettings: max_pages={settings.pdf_parser.max_pages}, "
      f"max_file_size_mb={settings.pdf_parser.max_file_size_mb}, "
      f"timeout={settings.pdf_parser.timeout}")
print("\n\u2713 Config integration confirmed")

PDFParserSettings: max_pages=30, max_file_size_mb=50, timeout=120

✓ Config integration confirmed


## 5. Exception Hierarchy

In [6]:
from src.exceptions import PaperAlchemyError, ParsingError, PDFParsingError, PDFValidationError

assert issubclass(PDFValidationError, PDFParsingError)
assert issubclass(PDFParsingError, ParsingError)
assert issubclass(ParsingError, PaperAlchemyError)

err = PDFValidationError("test error")
assert err.status_code == 422
print(f"PDFValidationError status_code: {err.status_code}")
print(f"Hierarchy: PDFValidationError -> PDFParsingError -> ParsingError -> PaperAlchemyError")
print("\n\u2713 Exception hierarchy verified")

PDFValidationError status_code: 422
Hierarchy: PDFValidationError -> PDFParsingError -> ParsingError -> PaperAlchemyError

✓ Exception hierarchy verified
